# Figure 1: Canopy Height Variation

Canopy height difference maps (GEDI vs ICESat) and PFT-level height boxplots.

PFT structure loaded from 5-year averaged data in `G:/Hangkai/CTH_ET project/Final_data/`

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import netCDF4
import os
from scipy.ndimage import zoom
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import cartopy.feature as cfeature
import pandas as pd
import seaborn as sns
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.gridspec as gridspec

# Custom colormap with white at center
coolwarm_cmap = plt.get_cmap('coolwarm')
colors_cm = coolwarm_cmap(np.linspace(0, 1, 200))
colors_cm[95:105] = (1, 1, 1, 1)
white_center_cmap = ListedColormap(colors_cm)
white_center_cmap.set_bad("white")

def plot_subimage(ax, file_path, cmap, vmin, vmax, multiply_by, zoom_factor=1,
                  show_top_labels=False, show_left_labels=True):
    with rasterio.open(file_path) as src:
        data = np.float32(src.read(1)) * multiply_by
        data[data <= -20] = -20
        data[data >= 20] = 20
        data = zoom(data, zoom_factor)

        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                          linewidth=0.5, color='gray', alpha=0.5,
                          xlocs=[-120, 0, 120], ylocs=[-90, -45, 0, 45, 90])
        labels = gl.xlabel_style; labels['size'] = 20; gl.xlabel_style = labels
        labels = gl.ylabel_style; labels['size'] = 20; gl.ylabel_style = labels
        gl.bottom_labels = False
        gl.top_labels = show_top_labels
        gl.left_labels = show_left_labels
        gl.right_labels = False
        ax.coastlines(linewidth=0.5)
        ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree())

# ============================================================
# Load PFT structure from Final_data (for height boxplot)
# ============================================================
file_path_final = r'G:\Hangkai\CTH_ET project\Final_data'
with netCDF4.Dataset(os.path.join(file_path_final, 'CLM Default.nc'), 'r') as ds:
    pfts1d_itype_veg = np.array(ds.variables['pfts1d_itype_veg'][:])
    pfts1d_ixy       = np.array(ds.variables['pfts1d_ixy'][:])
    pfts1d_jxy       = np.array(ds.variables['pfts1d_jxy'][:])
    pfts1d_wtgcell   = np.array(ds.variables['pfts1d_wtgcell'][:])

if pfts1d_itype_veg.ndim > 1:
    pfts1d_itype_veg = pfts1d_itype_veg[0, :]
    pfts1d_ixy = pfts1d_ixy[0, :]
    pfts1d_jxy = pfts1d_jxy[0, :]
    pfts1d_wtgcell = pfts1d_wtgcell[0, :]

pft_names = {
    "NET Temperate": 1, "NET Boreal": 2, "NDT Boreal": 3,
    "BET Tropical": 4, "BET Temperate": 5,
    "BDT Tropical": 6, "BDT Temperate": 7, "BDT Boreal": 8,
}
pft_mapping = {i: i + 1 for i in range(2, 10)}

# Load canopy height input data
file_path_height = r'H:\CLM_input\data'
scenarios = ['Default', 'Max', 'Mean', 'Median']
variable = 'MONTHLY_HEIGHT_TOP'

height_data = {}
for scenario in scenarios:
    nc_file = os.path.join(file_path_height, f"{scenario}.nc")
    with netCDF4.Dataset(nc_file, 'r') as ds:
        height_data[scenario] = ds.variables[variable][:]

# Compute weighted heights
weighted_heights = {}
for scenario in scenarios:
    weighted_heights[scenario] = {}
    for pft in pft_mapping:
        pft_indexes = np.where(pfts1d_itype_veg == pft_mapping[pft])[0]
        weight = pfts1d_wtgcell[pft_indexes]
        monthly_height = height_data[scenario][:, pft - 1,
                                                pfts1d_jxy[pft_indexes] - 1,
                                                pfts1d_ixy[pft_indexes] - 1]
        weighted_monthly_height = np.average(monthly_height, axis=1, weights=weight)
        weighted_heights[scenario][pft] = weighted_monthly_height

# Prepare boxplot data
box_data = []
for scenario in scenarios:
    for pft in pft_mapping:
        pft_indexes = np.where(pfts1d_itype_veg == pft_mapping[pft])[0]
        monthly_height = height_data[scenario][:, pft - 1,
                                                pfts1d_jxy[pft_indexes] - 1,
                                                pfts1d_ixy[pft_indexes] - 1]
        for height in monthly_height.flatten():
            box_data.append({'Scenario': scenario, 'PFT': pft, 'Height': height})

box_df = pd.DataFrame(box_data)
print('Data loaded.')

In [ ]:
# ============================================================
# Figure 1: Canopy height maps + PFT boxplot
# ============================================================
fig = plt.figure(figsize=(16.5, 10), dpi=300)
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1])
projection = ccrs.PlateCarree(central_longitude=180)

# (a) GEDI Max vs ICESat
ax1 = fig.add_subplot(gs[0, 0], projection=projection)
plot_subimage(ax1, r'H:\CTH_final_map\final_map\max_d.tif',
              white_center_cmap, -20, 20, 100)
cax1 = plt.colorbar(ScalarMappable(cmap='coolwarm', norm=Normalize(vmin=-20, vmax=20)),
                    ax=ax1, orientation='horizontal', shrink=0.4, pad=0.01)
cax1.set_label('GEDI Max vs ICESat (%)', fontsize=20)
cax1.ax.tick_params(labelsize=15)
ax1.coastlines()
ax1.text(-0.06, 1.0, '(a)', transform=ax1.transAxes, fontsize=16, fontweight='bold', va='top')

# (b) GEDI Mean vs ICESat
ax2 = fig.add_subplot(gs[0, 1], projection=projection)
plot_subimage(ax2, r'H:\CTH_final_map\final_map\mean_d.tif',
              white_center_cmap, -20, 20, 100, show_top_labels=True)
cax2 = plt.colorbar(ScalarMappable(cmap='coolwarm', norm=Normalize(vmin=-20, vmax=20)),
                    ax=ax2, orientation='horizontal', shrink=0.4, pad=0.01)
cax2.set_label('GEDI Mean vs ICESat (%)', fontsize=20)
cax2.ax.tick_params(labelsize=15)
ax2.coastlines()
ax2.text(-0.06, 1.0, '(b)', transform=ax2.transAxes, fontsize=16, fontweight='bold', va='top')

# (c) GEDI Median vs ICESat
ax3 = fig.add_subplot(gs[1, 0], projection=projection)
plot_subimage(ax3, r'H:\CTH_final_map\final_map\median_d.tif',
              white_center_cmap, -20, 20, 100)
cax3 = plt.colorbar(ScalarMappable(cmap='coolwarm', norm=Normalize(vmin=-20, vmax=20)),
                    ax=ax3, orientation='horizontal', shrink=0.4, pad=0.01)
cax3.set_label('GEDI Median vs ICESat (%)', fontsize=20)
cax3.ax.tick_params(labelsize=15)
ax3.coastlines()
ax3.text(-0.06, 1.0, '(c)', transform=ax3.transAxes, fontsize=16, fontweight='bold', va='top')

# (d) Canopy height boxplot by PFT
outer_ax = fig.add_subplot(gs[1, 1])
outer_ax.axis("off")
ax_box = inset_axes(outer_ax, width="96%", height="80%", loc='upper right')

color_palette = sns.color_palette("pastel", n_colors=len(scenarios))
sns.boxplot(x='PFT', y='Height', hue='Scenario', data=box_df,
            flierprops={"marker": "."}, palette=color_palette, fliersize=1, ax=ax_box)

# Add weighted average markers
for i, scenario in enumerate(scenarios):
    for j, pft in enumerate(pft_mapping):
        annual_weighted_height = np.mean(weighted_heights[scenario][pft], axis=0)
        ax_box.scatter(j + i * 0.2 - 0.3, annual_weighted_height,
                       marker='.', s=20, color='gray')

custom_labels = ['ICESat', 'GEDI Max', 'GEDI Mean', 'GEDI Median']
legend = ax_box.legend(handles=ax_box.get_legend().legendHandles,
                       labels=custom_labels,
                       bbox_to_anchor=(0.32, 1.01), loc='upper right',
                       handlelength=1.5, handletextpad=0.5, fontsize=15)

ax_box.set_xlabel("PFT", fontsize=0, color="white")
ax_box.set_ylabel("Height (m)", fontsize=16)
ax_box.set_xticks(np.arange(len(pft_mapping)))
ax_box.set_xticklabels(['NET Temp', 'NET Bor', 'NDT Bor', 'BET Trop', 'BET Temp',
                        'BDT Trop', 'BDT Temp', 'BDT Bor'], rotation=20, fontsize=16)
for label in ax_box.get_yticklabels():
    label.set_fontsize(14)
ax_box.text(-0.06, 1.00, '(d)', transform=ax_box.transAxes, fontsize=16,
            fontweight='bold', va='top', ha='left')

plt.tight_layout()
plt.savefig('figures/Figure_1_Canopy_Height.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/Figure_1_Canopy_Height.tiff', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')